# Verificar el paquete de desarrollo en Colab

Esta libreta comprueba la integridad y la estructura del conjunto de desarrollo preparado. Usa CPU y conserva el IMC faltante. No entrena modelos ni abre el paquete reservado de prueba.

Ejecuta las celdas en orden. Cuando se solicite, indica la carpeta de Drive que contiene `handoff.json`, el paquete de código y el paquete de desarrollo.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, stat, sys, tempfile, zipfile

drive.mount('/content/drive')
PACKAGE_DIR = Path(input('Carpeta del paquete en Drive: ').strip()).absolute()
if not PACKAGE_DIR.is_dir() or any(p.is_symlink() for p in (PACKAGE_DIR, *PACKAGE_DIR.parents)):
    raise ValueError('La carpeta indicada no existe o contiene enlaces.')


In [ ]:
def file_sha(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def require(condition, message):
    if not condition:
        raise ValueError(message)

handoff_path = PACKAGE_DIR / 'handoff.json'
require(handoff_path.is_file() and not handoff_path.is_symlink() and handoff_path.stat().st_size <= 1024 * 1024,
        'No se encontró un registro de entrega válido.')
handoff = json.loads(handoff_path.read_text(encoding='utf-8'))
require(handoff.get('version') == 'participant_split_colab_preparation_v1.0'
        and handoff.get('training_executed') is False
        and handoff.get('reserved_test_predictive_evaluation_opened') is False,
        'El registro no corresponde a la preparación autorizada.')
require(handoff['development']['file'] == 'development_package.zip'
        and handoff['reserved_test']['file'] == 'reserved_test_package.zip'
        and handoff['code']['file'] == 'colab_preparation_code.zip',
        'Los nombres de los paquetes no coinciden con la entrega.')

ALLOWED_CODE_FILES = {
    'src/knee/__init__.py', 'src/knee/colab_data.py', 'src/knee/colab_data_package.py',
    'src/knee/participant_partitions.py', 'scripts/verify_colab_data.py',
    'configs/participant_split_v1.json', 'requirements-colab-data.txt'}
require(set(handoff['code']['files_sha256']) == ALLOWED_CODE_FILES,
        'La lista de archivos de código no coincide.')
code_path = PACKAGE_DIR / handoff['code']['file']
require(code_path.is_file() and not code_path.is_symlink()
        and file_sha(code_path) == handoff['code']['sha256'], 'El paquete de código cambió.')
CODE_DIRECTORY = tempfile.TemporaryDirectory(prefix='knee_colab_data_')
code_root = Path(CODE_DIRECTORY.name)
with zipfile.ZipFile(code_path) as archive:
    entries = archive.infolist()
    require(len(entries) == len(ALLOWED_CODE_FILES)
            and {e.filename for e in entries} == ALLOWED_CODE_FILES, 'El contenido del código no coincide.')
    for entry in entries:
        require(not entry.is_dir() and 0 < entry.file_size <= 1024 * 1024
                and not stat.S_ISLNK(entry.external_attr >> 16)
                and stat.S_IFMT(entry.external_attr >> 16) in {0, stat.S_IFREG}
                and not entry.flag_bits & 1, 'El código contiene un archivo no permitido.')
        raw = archive.read(entry)
        require(hashlib.sha256(raw).hexdigest() == handoff['code']['files_sha256'][entry.filename],
                'La integridad de un archivo de código no coincide.')
        destination = code_root.joinpath(*entry.filename.split('/'))
        destination.parent.mkdir(parents=True, exist_ok=True)
        with destination.open('xb') as stream:
            stream.write(raw)
require(file_sha(code_path) == handoff['code']['sha256'], 'El código cambió durante su lectura.')
# Importación desde la copia verificada; el entorno no instala modelos ni PyTorch.
require('knee' not in sys.modules, 'Reinicia el entorno para evitar un módulo Knee cargado previamente.')
sys.path.insert(0, str(code_root / 'src'))
from knee.colab_data import verify_development_package
print('Código de verificación comprobado. Solo se abrirá el paquete de desarrollo.')

In [ ]:
development = handoff['development']
record_path = PACKAGE_DIR / 'colab_development_verification.json'
summary = verify_development_package(
    PACKAGE_DIR / development['file'], development['sha256'], record_path=record_path)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Comprobación guardada. No se entrenó ni se abrió el paquete reservado de prueba.')

Comparte únicamente el resumen final. El paquete reservado permanece separado. La imputación y las transformaciones específicas de cada modelo se ajustarán más adelante usando exclusivamente los datos de entrenamiento de cada pliegue.